# 03 — Baseline Models  (D4 Level 1 + D5)

**NovaFin Group Capstone · ePGD MLDS, IIIT Bombay · Group 2 · Dipesh Kumar Yadav**

Level 1 of the fine-tuning ladder: every declared model, cross-validated with
the Phase-2 splitters, logged to MLflow, ranked in a generated leaderboard.

**This notebook produces the first real numbers in the project.** Everything in
the guide and the deck marked `<<FILL AFTER RUN>>` is filled from here.

Three things make the numbers trustworthy:

1. **Everything is fitted inside the fold.** The pipeline carries the imputer,
   the scaler and the one-hot category list; `pipeline.fit(X_train)` runs per
   fold, so no transformation ever sees validation rows.
2. **Out-of-fold predictions are assembled, not averaged.** Each row is
   predicted once by a model that never trained on it. Thresholds, calibration
   curves and the permutation test are fitted to that single OOF vector.
3. **Fold variance travels with every headline.** Phase 2 showed per-fold AUC
   standard errors of 0.096 (initiatives) and 0.070 (churn). A bare mean would
   mislead.

> Prerequisites: notebooks `00`, `01`, `02`.

## 1 · Preamble

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "42"

IN_COLAB = "google.colab" in str(get_ipython())  # noqa: F821
if IN_COLAB:
    import subprocess, sys
    from pathlib import Path

    REPO_URL = "https://github.com/<YOUR-GITHUB-USERNAME>/novafin-capstone.git"
    if not Path("/content/novafin-capstone").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/novafin-capstone"], check=True)
    os.chdir("/content/novafin-capstone")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["NOVAFIN_DATA_RAW"] = (
        "/content/drive/MyDrive/ePGD - MLDS IIT Bombay/C5 ML In Finanace/Data"
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from novafin import viz
from novafin.config import load_config
from novafin.data import load_all, make_feature_frame, make_splitter, holdout_by_time
from novafin.evaluate import (
    bootstrap_ci, calibration_table, expected_calibration_error, expected_credit_loss,
    gains_table, optimal_threshold, permutation_test, population_stability_index,
    rank_ic, roc_auc,
)
from novafin.features import build_features
from novafin.models import (
    build_leaderboard, cross_validate_model, evaluate_cv, format_leaderboard,
    load_model_specs, make_bundle, train_module,
)
from novafin.tracking import ExperimentTracker, load_runs
from novafin.utils.logging_utils import setup_logging
from novafin.utils.seed import seed_everything
from novafin.utils.theme import apply_theme, save_figure

cfg = load_config()
setup_logging("WARNING", log_file=cfg.paths.logs / "03_baselines.log")
seed_everything(cfg.reproducibility.seed)
apply_theme()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)

tracker = ExperimentTracker(cfg)
print("config fingerprint:", cfg.fingerprint())
print("tracking backend  :", tracker.backend)

## 2 · Rebuild the feature matrices

Rebuilt rather than loaded from `data/processed/`, so this notebook is
self-contained and the features are guaranteed to match the current config
fingerprint.

In [ ]:
results = load_all(cfg=cfg)
features, matrices = {}, {}

for key, loaded in results.items():
    built = build_features(key, loaded.frame, cfg)
    features[key] = built
    spec = cfg.dataset(key)
    extra = [c for c in ("fwd_return_5d", "fwd_inflows_5d") if c in built.frame.columns]
    X, y = make_feature_frame(built.frame, spec, extra_drop=extra)
    X = X.drop(columns=[c for c in X.columns
                        if pd.api.types.is_datetime64_any_dtype(X[c])], errors="ignore")
    matrices[key] = (X, built.frame[built.target])

pd.DataFrame([
    {"module": k, "rows": len(X), "features": X.shape[1], "target": y.name}
    for k, (X, y) in matrices.items()
])

In [ ]:
# Data provenance travels with every artefact saved below.
DATA_HASHES = {r.path.name: r.sha256 for r in results.values()}
for name, digest in DATA_HASHES.items():
    print(f"  {name:<28} {digest[:16]}")

## 3 · M2 Credit Risk

Ranked by **KS**, with **Brier** alongside. PD feeds `ECL = PD × LGD × EAD`, so
a well-ranked but badly-calibrated model produces the wrong money — which is
why calibration is reported next to discrimination, not instead of it.

In [ ]:
X_loans, y_loans = matrices["loans"]
splitter_loans, kwargs_loans = make_splitter("loans", features["loans"].frame, cfg=cfg)
catalog_loans = load_model_specs("loans")

print("declared models:", catalog_loans.names())
results_loans = train_module(
    "loans", X_loans, y_loans, splitter_loans, catalog_loans.specs,
    task="binary_classification", cfg=cfg, split_kwargs=kwargs_loans,
    defaults=catalog_loans.defaults, tracker=tracker,
)

board_loans = pd.DataFrame([r.to_row() for r in results_loans])
display(board_loans[["module", "model", "roc_auc", "pr_auc", "ks", "brier",
                     "cv_roc_auc_std", "cv_n_folds"]].round(4))

In [ ]:
# Per-fold detail for the best model - this is what shows the variance.
best_loans = max(results_loans, key=lambda r: r.metrics.get("ks", float("-inf")))
print(f"best by KS: {best_loans.model_name}")
display(best_loans.fold_metrics[["fold", "n_train", "n_test", "roc_auc", "pr_auc", "ks", "brier"]].round(4))

print("\nPhase-2 predicted per-fold AUC standard error: 0.022")
print(f"Observed fold-to-fold AUC std              : {best_loans.metrics.get('cv_roc_auc_std', float('nan')):.4f}")
print("(If these agree, the Hanley-McNeil power analysis is validated empirically.)")

In [ ]:
# Calibration - the metric that decides whether ECL is trustworthy.
# ECE is computed from each model's own out-of-fold calibration table, so it
# reflects the same predictions the ECL calculation below will use.
calib_rows = []
for r in results_loans:
    ece = float("nan")
    if r.calibration is not None and not r.calibration.empty:
        weights = r.calibration["n"] / r.calibration["n"].sum()
        ece = float((weights * r.calibration["gap"].abs()).sum())
    calib_rows.append({
        "model": r.model_name,
        "roc_auc": r.metrics.get("roc_auc", float("nan")),
        "ks": r.metrics.get("ks", float("nan")),
        "brier": r.metrics.get("brier", float("nan")),
        "ece": ece,
    })
display(pd.DataFrame(calib_rows).round(4))
print("A model can rank well (high KS) and still be badly calibrated (high ECE).")
print("ECL multiplies the PROBABILITY by money, so ECE is the metric that decides")
print("whether the expected-loss numbers below are trustworthy.")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
if best_loans.calibration is not None:
    cal = best_loans.calibration
    ax.plot([0, cal["mean_predicted"].max()], [0, cal["mean_predicted"].max()],
            ls="--", lw=1, color="#C7C9C8", label="perfect calibration")
    ax.plot(cal["mean_predicted"], cal["observed_rate"], marker="o", color="#00B2A9",
            label=best_loans.model_name)
    ax.set_xlabel("Mean predicted PD"); ax.set_ylabel("Observed default rate")
    ax.set_title("M2 - PD calibration (out of fold)")
    ax.legend()
save_figure(fig, "03_m02_calibration", close=False); plt.show()

In [ ]:
# Expected Credit Loss, using the best model's out-of-fold PDs.
loans_frame = features["loans"].frame
pd_oof = pd.Series(np.nan, index=loans_frame.index)
# Recompute OOF predictions for the chosen model so PDs align row-for-row.
cv_best = cross_validate_model(
    catalog_loans.get(best_loans.model_name), X_loans, y_loans, splitter_loans,
    task="binary_classification", module="loans", cfg=cfg,
    split_kwargs=kwargs_loans, defaults=catalog_loans.defaults,
)
pd_oof[:] = cv_best.oof_predictions

lgd = cfg.fin("credit", "lgd", default=0.40)
ecl_flat = expected_credit_loss(pd_oof, loans_frame["Loan_Amount"], lgd=lgd)
ecl_collateral = expected_credit_loss(
    pd_oof, loans_frame["Loan_Amount"], collateral=loans_frame["Collateral_Value"]
)

bands = cfg.fin("credit", "pd_bands", default={})
def band_of(p):
    for name, (lo, hi) in bands.items():
        if lo <= p < hi:
            return name
    return "very_high"

summary = pd.DataFrame({
    "PD": pd_oof, "ECL_flat_lgd": ecl_flat, "ECL_collateral": ecl_collateral,
    "Loan_Amount": loans_frame["Loan_Amount"], "Default_Flag": y_loans,
    "risk_band": pd_oof.map(band_of),
})
by_band = summary.groupby("risk_band", observed=True).agg(
    n=("PD", "size"), mean_pd=("PD", "mean"), actual_default_rate=("Default_Flag", "mean"),
    exposure=("Loan_Amount", "sum"), ecl_flat=("ECL_flat_lgd", "sum"),
    ecl_collateral=("ECL_collateral", "sum"),
).round(4)
display(by_band)

print(f"\nTotal exposure        : {summary['Loan_Amount'].sum():,.0f}")
print(f"ECL at flat LGD {lgd:.0%}   : {ecl_flat.sum():,.0f}  ({ecl_flat.sum()/summary['Loan_Amount'].sum():.2%} of book)")
print(f"ECL collateral-aware  : {ecl_collateral.sum():,.0f}  ({ecl_collateral.sum()/summary['Loan_Amount'].sum():.2%} of book)")
print("\n'mean_pd' vs 'actual_default_rate' per band IS the calibration check that matters for ECL.")

## 4 · M3 Fraud — the cost-optimal threshold

The brief's economics, applied directly: a missed fraud costs ₹10,000, an
unnecessary investigation ₹500. **The decision variable is the threshold, not
the model** — and the optimum is emphatically not 0.5.

Both do-nothing baselines are reported. A model earns its keep only by beating
"flag nothing" *and* "flag everything".

In [ ]:
X_txn, y_txn = matrices["transactions"]
txn_frame = features["transactions"].frame

# Chronological holdout: 2026-06-01 onwards is scored ONCE, at the very end.
train_idx, holdout_idx = holdout_by_time(txn_frame, "transactions", cfg=cfg)
print(f"train  : {len(train_idx):,} rows to {txn_frame['Timestamp'].iloc[train_idx].max().date()}")
print(f"holdout: {len(holdout_idx):,} rows from {txn_frame['Timestamp'].iloc[holdout_idx].min().date()}")

splitter_txn, kwargs_txn = make_splitter("transactions", txn_frame, cfg=cfg)
catalog_txn = load_model_specs("transactions")

results_txn = train_module(
    "transactions", X_txn.iloc[train_idx], y_txn.iloc[train_idx], splitter_txn,
    [s for s in catalog_txn.specs if not s.unsupervised],
    task="binary_classification", cfg=cfg, defaults=catalog_txn.defaults, tracker=tracker,
    cost_false_negative=cfg.fin("fraud", "cost_missed_fraud_inr", default=10000),
    cost_false_positive=cfg.fin("fraud", "cost_false_positive_inr", default=500),
)

board_txn = pd.DataFrame([r.to_row() for r in results_txn])
display(board_txn[["model", "roc_auc", "pr_auc", "brier", "optimal_threshold",
                   "expected_cost", "cv_pr_auc_std"]].round(4))

In [ ]:
best_txn = max(results_txn, key=lambda r: r.metrics.get("pr_auc", float("-inf")))
print(f"best by PR-AUC: {best_txn.model_name}")
print("\nCOST ECONOMICS (out of fold, training period)")
for k, v in best_txn.cost.items():
    print(f"   {k:<26} {v:,.2f}")

cost_fn = cfg.fin("fraud", "cost_missed_fraud_inr", default=10000)
cost_fp = cfg.fin("fraud", "cost_false_positive_inr", default=500)
saving = best_txn.cost["saving_vs_best_baseline"]
print(f"\nThe model saves {saving:,.0f} INR against the better do-nothing policy.")
print(f"At the optimal threshold of {best_txn.cost['threshold']:.3f}, the team investigates "
      f"{best_txn.cost['n_flagged']:,.0f} transactions and catches {best_txn.cost['recall']:.1%} of fraud.")

In [ ]:
# The cost curve - the figure that makes the threshold decision visible.
from novafin.evaluate import cost_curve

cv_txn = cross_validate_model(
    catalog_txn.get(best_txn.model_name), X_txn.iloc[train_idx], y_txn.iloc[train_idx],
    splitter_txn, task="binary_classification", module="transactions", cfg=cfg,
    defaults=catalog_txn.defaults,
)
oof_mask = cv_txn.oof_mask
curve = cost_curve(
    np.asarray(y_txn.iloc[train_idx])[oof_mask],
    np.asarray(cv_txn.oof_predictions)[oof_mask],
    cost_false_negative=cost_fn, cost_false_positive=cost_fp,
)

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(curve["threshold"], curve["total_cost"], color="#00B2A9", lw=2)
best_row = curve.loc[curve["total_cost"].idxmin()]
ax.axvline(best_row["threshold"], color="#B88F20", ls="--", lw=1.2,
           label=f"optimum {best_row['threshold']:.3f}")
ax.axvline(0.5, color="#C7C9C8", ls=":", lw=1.2, label="naive 0.5")
ax.set_xlabel("Threshold"); ax.set_ylabel("Expected cost (INR)")
ax.set_title("M3 - expected operational cost vs threshold")
ax.legend()
save_figure(fig, "03_m03_cost_curve", close=False); plt.show()

naive_cost = float(curve.loc[(curve["threshold"] - 0.5).abs().idxmin(), "total_cost"])
print(f"cost at the naive 0.5 threshold : {naive_cost:,.0f}")
print(f"cost at the optimal threshold   : {best_row['total_cost']:,.0f}")
print(f"saving from thresholding alone  : {naive_cost - best_row['total_cost']:,.0f}")

> **The 2026-06 to 2026-08 holdout stays unscored.** It is touched once, in
`06_evaluation_and_explainability`, after every tuning decision has been made.
Scoring it now - even "just to look" - would make it a validation set.

## 5 · M4 Churn — the designed negative result

Phase 0 found 89 positives and a maximum feature correlation of 0.0225.
Phase 2 computed a per-fold AUC standard error of 0.070 — a 95% interval of
roughly ±0.14.

In that setting a cross-validated AUC of, say, 0.56 is **not** evidence of
signal. The permutation test makes that precise: refit the model on shuffled
targets many times and ask how often the null beats the observed score.

In [ ]:
X_cust, y_cust = matrices["customers"]
splitter_cust, kwargs_cust = make_splitter("customers", features["customers"].frame, cfg=cfg)
catalog_cust = load_model_specs("customers")

results_cust = train_module(
    "customers", X_cust, y_cust, splitter_cust, catalog_cust.specs,
    task="binary_classification", cfg=cfg, defaults=catalog_cust.defaults, tracker=tracker,
)
board_cust = pd.DataFrame([r.to_row() for r in results_cust])
display(board_cust[["model", "roc_auc", "pr_auc", "brier", "cv_roc_auc_std", "cv_n_folds"]].round(4))

In [ ]:
# THE PERMUTATION TEST. `fit_predict` must REFIT on whatever target it is
# given - reusing predictions from the true target would invalidate the test.
spec_cust = catalog_cust.get("lightgbm")

def fit_predict_churn(target):
    result = cross_validate_model(
        spec_cust, X_cust, pd.Series(target, name="Churn_Flag"), splitter_cust,
        task="binary_classification", module="customers", cfg=cfg,
        defaults=catalog_cust.defaults,
    )
    return np.asarray(result.oof_predictions)

N_PERMUTATIONS = 100      # raise to 200+ for the final report if time allows
churn_test = permutation_test(
    fit_predict_churn, np.asarray(y_cust),
    n_permutations=N_PERMUTATIONS, seed=cfg.reproducibility.seed,
)
print(churn_test)
display(pd.DataFrame([churn_test.summary()]).round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(churn_test.null_scores, bins=30, color="#C7C9C8", label="null (shuffled target)")
ax.axvline(churn_test.observed, color="#B88F20", lw=2.5,
           label=f"observed {churn_test.observed:.4f}")
ax.axvline(churn_test.null_p95, color="#00B2A9", ls="--", lw=1.5,
           label=f"null p95 {churn_test.null_p95:.4f}")
ax.set_xlabel("Cross-validated ROC-AUC"); ax.set_ylabel("Count")
ax.set_title(f"M4 - permutation test, p = {churn_test.p_value:.4f}")
ax.legend()
save_figure(fig, "03_m04_permutation_test", close=False); plt.show()

verdict = ("NO evidence of learnable churn signal" if churn_test.p_value >= 0.05
           else "evidence of signal - re-examine finding N-01")
print(f"VERDICT: {verdict}")
print("\nThe executive question is answered on VALUE instead - see notebook 02 section 6.2.")

## 6 · M5/M6 Equity — ranked by information coefficient

Scored with **rank IC**, computed within each rebalance date and then averaged.
Nobody trades a return forecast; they trade its cross-sectional ordering.
Pooling all dates into one correlation would mix cross-sectional skill with the
time-series level of returns.

In [ ]:
X_mkt, y_mkt = matrices["market"]
market_frame = features["market"].frame
splitter_mkt, kwargs_mkt = make_splitter("market", market_frame, cfg=cfg)
catalog_mkt = load_model_specs("market")

results_mkt = train_module(
    "market", X_mkt, y_mkt, splitter_mkt, catalog_mkt.specs,
    task="panel_regression", cfg=cfg, split_kwargs=kwargs_mkt,
    defaults=catalog_mkt.defaults, ic_groups=market_frame["Date"], tracker=tracker,
)
board_mkt = pd.DataFrame([r.to_row() for r in results_mkt])
display(board_mkt[["model", "rmse", "mae", "r2", "cv_mean_ic", "cv_ic_ir", "cv_hit_rate"]].round(5)
        if "cv_mean_ic" in board_mkt.columns else board_mkt.round(5))

In [ ]:
best_mkt = max(results_mkt, key=lambda r: r.metrics.get("cv_mean_ic", float("-inf")))
print(f"best by mean IC: {best_mkt.model_name}")
display(best_mkt.fold_metrics.round(5))
print("\nA daily equity mean IC of 0.02-0.05 is a genuinely useful signal;")
print("anything above ~0.15 on daily data should be treated as a leak until proven otherwise.")

## 7 · M9 HFT and M10 Derivatives

In [ ]:
# --- M9: 3-class, macro-F1 (FLAT is only 16.4% of rows) -------------------
X_hft, y_hft = matrices["hft"]
hft_frame = features["hft"].frame
splitter_hft, kwargs_hft = make_splitter("hft", hft_frame, cfg=cfg)
catalog_hft = load_model_specs("hft")

results_hft = train_module(
    "hft", X_hft, y_hft, splitter_hft, catalog_hft.specs,
    task="multiclass_classification", cfg=cfg, split_kwargs=kwargs_hft,
    defaults=catalog_hft.defaults, tracker=tracker,
)
board_hft = pd.DataFrame([r.to_row() for r in results_hft])
display(board_hft[[c for c in ["model", "accuracy", "macro_f1", "balanced_accuracy",
                               "recall_UP", "recall_DOWN", "recall_FLAT"]
                   if c in board_hft.columns]].round(4))

In [ ]:
# --- M10: the FAIR FIGHT and the RESIDUAL model (register L-04) -----------
from novafin.features import PRIMITIVE_FEATURES

opt_frame = features["options"].frame
catalog_opt = load_model_specs("options")
splitter_opt, kwargs_opt = make_splitter("options", opt_frame, cfg=cfg)

# (a) Fair fight: ML sees exactly what Black-Scholes sees.
X_fair = opt_frame[PRIMITIVE_FEATURES].copy()
y_market = opt_frame["Market_Price"]
results_fair = train_module(
    "options", X_fair, y_market, splitter_opt, catalog_opt.specs,
    task="regression", cfg=cfg, split_kwargs=kwargs_opt,
    defaults=catalog_opt.defaults, tracker=tracker,
)
fair = pd.DataFrame([r.to_row() for r in results_fair])[["model", "rmse", "mae", "r2"]]

# The benchmark: Black-Scholes itself.
from novafin.evaluate import regression_metrics
bs_metrics = regression_metrics(y_market, opt_frame["Black_Scholes_Price"])
print("BLACK-SCHOLES benchmark:", {k: round(v, 4) for k, v in bs_metrics.items()})
display(fair.round(4))
print("\nThe question is not whether ML beats BS with BS as a feature (it trivially would),")
print("but whether ML can REDISCOVER the formula from the same primitives.")

In [ ]:
# (b) Residual model: target = Market_Price - Black_Scholes_Price
X_resid = opt_frame[PRIMITIVE_FEATURES + ["moneyness", "log_moneyness", "vol_sqrt_t",
                                          "standardised_moneyness", "d1", "d2",
                                          "vega", "gamma", "time_value"]].copy()
y_resid = opt_frame["mispricing"]
results_resid = train_module(
    "options_residual", X_resid, y_resid, splitter_opt, catalog_opt.specs,
    task="regression", cfg=cfg, split_kwargs=kwargs_opt,
    defaults=catalog_opt.defaults, tracker=tracker,
)
display(pd.DataFrame([r.to_row() for r in results_resid])[["model", "rmse", "mae", "r2"]].round(4))
print(f"\nStd of the mispricing target: {y_resid.std():.4f}")
print("An R^2 near zero here is the HONEST answer if the residual is pure noise -")
print("and the audit suggests it may be. Report it either way.")

## 8 · M1 Initiatives, M7 Liquidity, M4b CLV

In [ ]:
remaining = [
    ("initiatives", "binary_classification", None),
    ("liquidity", "time_series_forecast", None),
]
all_results = {"loans": results_loans, "transactions": results_txn,
               "customers": results_cust, "market": results_mkt, "hft": results_hft,
               "options": results_fair}

for key, task, groups in remaining:
    X, y = matrices[key]
    splitter, kwargs = make_splitter(key, features[key].frame, cfg=cfg)
    catalog = load_model_specs(key)
    res = train_module(key, X, y, splitter, catalog.specs, task=task, cfg=cfg,
                       split_kwargs=kwargs, defaults=catalog.defaults, tracker=tracker)
    all_results[key] = res
    print(f"\n=== {key.upper()} ===")
    display(pd.DataFrame([r.to_row() for r in res]).round(4))

In [ ]:
# M4b - CLV regression, framed as ATTRIBUTION not forecasting (register L-07).
cust_frame = features["customers"].frame
clv_exclude = {"Estimated_CLV", "Churn_Flag", "Customer_ID",
               "decision_value_rank", "decision_attrition_proxy", "decision_priority_score"}
X_clv = cust_frame.drop(columns=[c for c in clv_exclude if c in cust_frame.columns])
X_clv = X_clv.select_dtypes(include=[np.number, "object"])
y_clv = cust_frame["Estimated_CLV"]

catalog_clv = load_model_specs("customers_clv")
splitter_clv, kwargs_clv = make_splitter("customers", cust_frame, cfg=cfg)
results_clv = train_module("customers_clv", X_clv, y_clv, splitter_clv, catalog_clv.specs,
                           task="regression", cfg=cfg, defaults=catalog_clv.defaults,
                           tracker=tracker)
all_results["customers_clv"] = results_clv
display(pd.DataFrame([r.to_row() for r in results_clv])[["model", "rmse", "mae", "r2"]].round(3))
print("\nA very high R^2 here is EXPECTED and is not a success: Estimated_CLV is a")
print("formula over its own predictors. The value of this model is the SHAP")
print("attribution in notebook 06, not the score.")

## 9 · D5 — the generated leaderboard

Generated from the run store, never typed. The table in the D7 guide and on the
D8 results slide is produced by `python -m novafin.models.leaderboard`.

In [ ]:
runs = load_runs(cfg)
print(f"runs recorded: {len(runs)}")

board = build_leaderboard(cfg)
display(board.round(4))

leaderboard_path = cfg.paths.tables / "leaderboard.csv"
board.to_csv(leaderboard_path, index=False)
(cfg.paths.tables / "leaderboard.md").write_text(format_leaderboard(board), encoding="utf-8")
print(f"\nwritten: {leaderboard_path}")
print(format_leaderboard(board))

## 10 · Persist the champion bundles

Each bundle carries the model, its preprocessor, the exact feature list, the
cross-validated metrics, the config fingerprint and the SHA-256 of every input
file. A bundle that cannot say what it was trained on is not reproducible.

In [ ]:
champions = {
    "loans": ("loans", best_loans.model_name, catalog_loans, X_loans, y_loans),
    "transactions": ("transactions", best_txn.model_name, catalog_txn,
                     X_txn.iloc[train_idx], y_txn.iloc[train_idx]),
    "market": ("market", best_mkt.model_name, catalog_mkt, X_mkt, y_mkt),
}

cfg.paths.artifacts.mkdir(parents=True, exist_ok=True)
for key, (module, model_name, catalog, X, y) in champions.items():
    evaluation = max(all_results[module], key=lambda r: list(r.headline().values())[0]
                     if r.headline() else float("-inf"))
    bundle = make_bundle(
        catalog.get(model_name), X, y, evaluation, cfg=cfg,
        defaults=catalog.defaults, data_hashes=DATA_HASHES,
        metadata={"phase": "4_baseline", "level": "L1"},
    )
    path = bundle.save(cfg.paths.artifacts / f"{key}_baseline.pkl")
    print(f"  {path.name:<28} {len(bundle.feature_names):>3} features  "
          f"fingerprint {bundle.config_fingerprint}")

In [ ]:
# Summary table for the D7 guide and the README results section.
summary_rows = []
for module, res in all_results.items():
    for r in res:
        row = {"module": module, "model": r.model_name, "task": r.task}
        row.update({k: round(v, 5) for k, v in r.headline().items()})
        row["cv_folds"] = r.metrics.get("cv_n_folds", float("nan"))
        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(cfg.paths.tables / "03_baseline_summary.csv", index=False)
display(summary)
print("\nTables written:")
for path in sorted(cfg.paths.tables.glob("*.csv")):
    print("   ", path.name)

---

## Phase 4 summary

| Guarantee | Established by |
|---|---|
| No transformation sees validation rows | the pipeline is fitted inside each fold |
| Every row predicted by a model that did not train on it | out-of-fold assembly |
| Fold variance is visible | `cv_*_std` beside every headline metric |
| The churn null is tested, not asserted | permutation test with a refit per shuffle |
| Thresholds follow the money | cost curve with the brief's ₹10,000 / ₹500 |
| Results cannot drift from their runs | leaderboard generated from the run store |
| Every model beat a declared trivial baseline, or did not | dummy/mean/zero first in every module |

**NEXT:** `04_hyperparameter_tuning` — D4 Level 2: Optuna with median pruning,
a documented search-space rationale per module, and a **resumable SQLite study**
so a Colab disconnect costs you nothing.